# Healthcare Data Ingestion → Snowflake STAGING

This notebook downloads the Synthea dataset, extracts Batch 1, and loads the final selected CSV set into the Snowflake internal stage and STAGING tables.

**Tables loaded:** patients, encounters, observations, conditions, medications, allergies, procedures, immunizations, careplans.

The original ingestion logic is preserved while duplicated Snowflake code and unnecessary inspection/debug cells are consolidated.

## 1. Imports & Configuration

In [ ]:
import os
import shutil
import snowflake.connector
from pathlib import Path

# Local paths
DATA_DIR = Path('/content/syntheticmass')
ARCHIVE_PATH = Path('/content/dataset.tar.gz')
TEMP_DIR = Path('/content/temp_chunk')
BATCH_1_ARCHIVE = DATA_DIR / 'synthea_1m_fhir_3_0_May_24' / 'output_1_20170524T232103.tar.gz'
BATCH_1_CSV_DIR = TEMP_DIR / 'output_1' / 'csv'

# Snowflake settings
SNOWFLAKE_ACCOUNT = 'zj90931.eu-central-2.aws'
SNOWFLAKE_USER = 'ahmedSami'
SNOWFLAKE_WAREHOUSE = 'HEALTHCARE_WH'
SNOWFLAKE_DATABASE = 'HEALTHCARE_DB'
SNOWFLAKE_SCHEMA = 'STAGING'
SNOWFLAKE_STAGE = 'synthea_stage'

# Keep credentials outside the notebook.
# Set SNOWFLAKE_PASSWORD in the runtime environment before running.
SNOWFLAKE_PASSWORD = os.getenv('SNOWFLAKE_PASSWORD')
if not SNOWFLAKE_PASSWORD:
    raise RuntimeError('SNOWFLAKE_PASSWORD is not set in the runtime environment.')

# Final selected file set for the STAGING stage/tables.
STAGING_FILES = [
    'patients.csv',
    'encounters.csv',
    'observations.csv',
    'conditions.csv',
    'medications.csv',
    'allergies.csv',
    'procedures.csv',
    'immunizations.csv',
    'careplans.csv',
]

## 2. Download & Extract Dataset

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not ARCHIVE_PATH.exists():
    print('Downloading 21GB Dataset...')
    result = os.system(
        f'wget -L -O {ARCHIVE_PATH} '
        '"https://mitre.box.com/shared/static/3bo45m48ocpzp8fc0tp005vax7l93xji.gz"'
    )
    if result != 0:
        raise RuntimeError('Failed to download the dataset archive.')
else:
    print(f'Dataset archive already exists: {ARCHIVE_PATH}')

if not BATCH_1_ARCHIVE.exists():
    print('Extracting outer dataset archive...')
    result = os.system(f'tar -xzf {ARCHIVE_PATH} -C {DATA_DIR}')
    if result != 0:
        raise RuntimeError('Failed to extract the outer dataset archive.')
else:
    print('Outer dataset archive is already extracted.')

# Remove the large outer archive once extraction is complete.
if ARCHIVE_PATH.exists():
    ARCHIVE_PATH.unlink()
    print('Removed outer dataset archive.')

print(f'Batch 1 archive: {BATCH_1_ARCHIVE}')

## 3. Extract Batch 1 CSV Files

In [ ]:
# Clean previous temporary extraction so the selected files are always from the current Batch 1 archive.
if TEMP_DIR.exists():
    shutil.rmtree(TEMP_DIR)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

print('Extracting Batch 1...')
result = os.system(f'tar -xzf {BATCH_1_ARCHIVE} -C {TEMP_DIR}')
if result != 0:
    raise RuntimeError('Failed to extract Batch 1 archive.')

missing_files = [
    name for name in STAGING_FILES
    if not (BATCH_1_CSV_DIR / name).exists()
]
if missing_files:
    raise FileNotFoundError(
        'Expected CSV files are missing from Batch 1: ' + ', '.join(missing_files)
    )

print(f'Found all {len(STAGING_FILES)} required CSV files:')
for name in STAGING_FILES:
    print(f'  ✓ {name}')

## 4. Upload to Snowflake STAGING Stage & Tables

In [ ]:
# One connection and one upload/load loop for all selected tables.
# This replaces the two duplicated Snowflake blocks in the original notebook.

conn = snowflake.connector.connect(
    user=SNOWFLAKE_USER,
    password=SNOWFLAKE_PASSWORD,
    account=SNOWFLAKE_ACCOUNT,
    warehouse=SNOWFLAKE_WAREHOUSE,
    database=SNOWFLAKE_DATABASE,
    schema=SNOWFLAKE_SCHEMA,
)

try:
    with conn.cursor() as cursor:
        cursor.execute(f'CREATE STAGE IF NOT EXISTS {SNOWFLAKE_STAGE};')
        print(f'Using Snowflake stage: @{SNOWFLAKE_STAGE}')

        for file_name in STAGING_FILES:
            local_file_path = BATCH_1_CSV_DIR / file_name
            table_name = Path(file_name).stem.upper()

            print(f'\nUploading {file_name}...')
            cursor.execute(
                f'PUT file://{local_file_path} @{SNOWFLAKE_STAGE} '
                'AUTO_COMPRESS=TRUE OVERWRITE=TRUE;'
            )

            print(f'Loading {table_name}...')
            copy_query = f"""
                COPY INTO {table_name}
                FROM @{SNOWFLAKE_STAGE}/{file_name}.gz
                FILE_FORMAT = (
                    TYPE = CSV
                    SKIP_HEADER = 1
                    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
                    NULL_IF = ('')
                )
                ON_ERROR = 'CONTINUE';
            """
            cursor.execute(copy_query)
            print(f'✓ Done: {table_name}')

    print('\nAll selected Synthea tables loaded successfully into STAGING.')

except Exception as e:
    print(f'Error during Snowflake ingestion: {e}')
    raise
finally:
    conn.close()

## 5. Final Validation

In [ ]:
conn = snowflake.connector.connect(
    user=SNOWFLAKE_USER,
    password=SNOWFLAKE_PASSWORD,
    account=SNOWFLAKE_ACCOUNT,
    warehouse=SNOWFLAKE_WAREHOUSE,
    database=SNOWFLAKE_DATABASE,
    schema=SNOWFLAKE_SCHEMA,
)

try:
    with conn.cursor() as cursor:
        for file_name in STAGING_FILES:
            table_name = Path(file_name).stem.upper()
            cursor.execute(f'SELECT COUNT(*) FROM {table_name};')
            row_count = cursor.fetchone()[0]
            print(f'{table_name}: {row_count:,} rows')
finally:
    conn.close()